# System Architecture, Design & Containers · Edge-Computing Notebook

Before you can design an edge system, you have to understand the device it runs on. An edge architecture is shaped by the resources of the edge node: how many CPU cores it has, how much memory and storage, what accelerator it carries, how much power it draws, and how much of all of that is already in use.

In this lab you will **profile a DGX Spark as an edge node** — measuring its real hardware and live load — and then use those numbers to make architecture decisions: what runs on the device, what runs in the cloud, and how many services the device can realistically host. Finally, you turn the design into something that actually runs by **containerizing the edge services with Docker** — including a GPU container — so the paper design becomes running software on the node.

```text
Measure the edge node  ->  Understand its limits  ->  Design what fits  ->  Containerize it
```

The main idea is edge architecture is **constraint-driven design**. The device's measured resources are the constraints, good design lives within them, and Docker is how those designed services get packaged and run.

## How this notebook works · cells and a Jupyter terminal

Most steps in this lab run once and return (reading a spec, building an image) — those are ordinary notebook cells. A few steps are live views that never return on their own (`tegrastats`, live `docker stats`); for those the notebook writes a small script into your lab folder and asks you to run it in a Jupyter terminal.

Two labels are used throughout:

- **[Notebook cell]** · run the cell right here.
- **[Terminal]** · open a Jupyter terminal — **File ▸ New ▸ Terminal** (or the blue **+** Launcher, then the **Terminal** tile) — and run the given command there. **Ctrl-C** stops a live view. You can open several terminals at once.

### One config, everywhere

Terminal shells do not share the notebook kernel's variables. So the setup cell below writes `~/edgeArchitectureLab/labEnv.sh` with your `PORT` and `NVIDIA_IMAGE`, and every helper script starts with `source ~/edgeArchitectureLab/labEnv.sh` — the notebook and your terminals always agree with no manual step.

**Requirements:** run this notebook with a Python 3 kernel on the DGX Spark, where Docker is available to your user. Parts 16-17 additionally need the NVIDIA runtime.

In [ ]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, preflight checks, and checkpoints.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeClassHandson"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
from labHelpers import *

---
## Part 0 · Before You Begin

You are working on a shared **DGX Spark** edge device, already connected with your own student account (set up in the *SSH, Linux, Git, GitHub* lab). **Every command in this exercise runs on the DGX Spark** — this notebook runs there through JupyterLab.

Because the DGX Spark is shared:

- use your own home folder and keep your files under it
- prefix any shared resource names (containers, images, volumes, Compose projects) with your account name using `${USER}`
- use only the ports assigned to you

The Docker parts later in this lab (Part 11 onward) run through the same Docker daemon on the DGX Spark, so containers, images, and host ports are shared across the whole device. The `${USER}` prefix and your assigned port keep your work separate from everyone else's.

**[Notebook cell]** The original lab asks you to `export PORT=<your-port>` by hand in Part 11. Here your port is derived automatically from the digits in your username (`student07` → `5007`), so nobody collides and you normally edit nothing — to force a specific port, pass `portOverrides={"PORT": 5123}`. The cell also writes `labEnv.sh` for the terminal helper scripts and sets `NVIDIA_IMAGE` for Parts 16-17 (default targets DGX Spark, JetPack 7 / CUDA 13 — use the tag your instructor provides if it differs):

In [ ]:
labConfig = setupLab(
    "edgeArchitectureLab",
    ports={"PORT": 5000},
    extraEnv={"NVIDIA_IMAGE": "nvcr.io/nvidia/cuda:13.0.0-devel-ubuntu24.04"},
)

### Preflight · check your environment

**[Notebook cell]** Run this once at the start. Parts 1-10 need only Linux tools; Parts 11-18 need Docker; Parts 16-17 need the NVIDIA runtime. Fix what you can before continuing:

In [ ]:
import os
preflight([
    check("docker daemon", dockerDaemonUp(),
          hint="Docker must be running and your account must have an instructor-provisioned rootless Podman socket - "
               "ask your instructor. Needed from Part 11 onward.",
          link="https://docs.docker.com/engine/daemon/troubleshoot/",
          linkText="Docker daemon troubleshooting"),
    check("docker compose", composeAvailable(),
          hint="Compose v2 ships with modern Docker; Parts 15 and 17 need it.",
          link="https://docs.docker.com/compose/install/",
          linkText="Installing Compose"),
    check("nvidia runtime", nvidiaRuntimeAvailable(),
          hint="Parts 16-17 need the NVIDIA container runtime; everything before them "
               "works without it. Ask your instructor if it is missing.",
          link="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html",
          linkText="NVIDIA Container Toolkit"),
    check("git installed", commandOnPath("git"),
          hint="Git is only needed for the optional commit in Part 10 - install it or "
               "skip that step.",
          link="https://git-scm.com/downloads",
          linkText="Git downloads"),
], infoRows=[("your USER", os.environ.get("USER", "?")),
             ("your PORT", os.environ.get("PORT", "?")),
             ("NVIDIA_IMAGE", os.environ.get("NVIDIA_IMAGE", "?"))])

---
## Part 1 · Set Up Your Working Folder

**[Notebook cell]** Create a folder to hold the profile and design files you produce in this lab. You will save measurements into `profile/` and write design documents at the top level:

In [ ]:
!mkdir -p ~/edgeArchitectureLab/profile
%cd ~/edgeArchitectureLab

---
## Part 2 · Identify the Edge Node

Never assume the device's specs — make the device report them itself.

> **New term — edge node:** the computer that does compute *near* where data is produced, between the sensors and the cloud. Here, the DGX Spark is the edge node.

**[Notebook cell]** Read the board model:

> **📟 On a Jetson:** device identity is read differently per platform. A Jetson is an ARM *device-tree* board, so its model lives in `/proc/device-tree/model`, and its OS is **L4T** (Linux for Tegra) tagged in `/etc/nv_tegra_release` / `jetson_release`. This **DGX Spark (GB10)** is a standard system: model comes from **DMI** (`/sys/class/dmi/id/*`) and it runs stock **Ubuntu**. Same questions — *what am I?* — different files.

In [ ]:
# Board / system model. A discrete box like this GB10 exposes it via DMI (SMBIOS);
# a Jetson exposes it via the ARM device-tree. Try DMI first, fall back to device-tree.
!cat /sys/class/dmi/id/product_name 2>/dev/null || cat /proc/device-tree/model 2>/dev/null || echo "model not exposed"
!nvidia-smi -L

**You should see:** the device's product name (a Jetson model string). This file has no trailing newline, which is why the cell adds an `echo`.

**[Notebook cell]** Check the operating system:

In [ ]:
!cat /etc/os-release
!uname -a

**[Notebook cell]** If the Jetson L4T release file is present, read it, and if the `jetson_release` tool is installed (part of the jetson-stats package), get its clean summary:

In [ ]:
# OS / release. Jetson ships "Linux for Tegra" (L4T) with JetPack; this DGX Spark
# runs stock Ubuntu, so we read the standard OS + kernel info instead.
!hostnamectl 2>/dev/null | grep -E "Operating System|Kernel|Hardware Vendor" || lsb_release -a 2>/dev/null
!cat /etc/nv_tegra_release 2>/dev/null || echo "(no L4T release file - not a Jetson)"


This is the first design fact: **what device am I architecting for?** (You capture all of these readings into one saved report in Part 9 — no need to save each one by hand.)

---
## Part 3 · Profile the CPU

**[Notebook cell]** Count the CPU cores, get detailed CPU information, and check the current load:

In [ ]:
!nproc
!uptime

In [ ]:
!lscpu

The three numbers after `load average` are the 1, 5, and 15 minute averages. Compare them to your core count from `nproc`:

```text
load average ~ number of cores  -> the CPU is fully busy
load average <  number of cores -> the CPU has headroom
```

> The DGX Spark is shared. The load average includes **every** student's work, not just yours. A high number does not mean your code is heavy — it means the device is busy. This is itself a core edge-architecture lesson: edge nodes have finite, shared compute.

---
## Part 4 · Profile Memory

**[Notebook cell]** Check total and available memory:

In [ ]:
!free -h

Key columns:

```text
total     -> all RAM on the device
used      -> RAM in use right now (by everyone, shared device)
available -> RAM that can be given to a new program
```

Memory is usually the **tightest** constraint on an edge node. A model or container that needs more RAM than is available simply will not run, so the available memory directly limits how many services you can place on the device.

---
## Part 5 · Profile the GPU and Accelerators

The whole point of a Jetson is its GPU/AI accelerator. Inspect it.

**[Notebook cell]** Check for NVIDIA device files, the CUDA install, and `nvidia-smi`:

In [ ]:
!ls /dev/nvhost* /dev/nvidia* 2>/dev/null | head || echo "No NVIDIA device files found"
!ls /usr/local/cuda 2>/dev/null && cat /usr/local/cuda/version.json 2>/dev/null | head || echo "CUDA folder not found"
!nvidia-smi 2>/dev/null || echo "nvidia-smi not available on this device"


> **If a command says "not found" or "not available":** that is expected here, not a failure. Tools like `nvidia-smi`, `jetson_release`, and `nvpmodel` aren't installed on every Jetson configuration. Each command above falls back to a clear message and the lab continues — just record "not available" in your profile and move on. The point of this lab is to capture what the device *does* report.

The accelerator is what makes AI inference possible at the edge. Whether a model runs **on the device** or must go **to the cloud** often comes down to what you find here.

---
## Part 6 · Profile Storage and I/O

**[Notebook cell]** Check disk space, the block devices, and the size of your own home folder:

In [ ]:
!df -h
!lsblk
!du -sh ~ 2>/dev/null

Storage limits how much local data the device can retain. Edge systems often cannot keep raw data forever, which is why later labs use retention policies and time-series databases.

> The disk is shared too. `df -h` shows space used by all accounts. Keep your own footprint small.

### Checkpoint · Parts 1-6

In [ ]:
checkpoint("Parts 1-6 - edge node profiled", [
    check("lab folder ready", dirExists("~/edgeArchitectureLab/profile"),
          hint="Re-run the Part 1 cell (mkdir -p ~/edgeArchitectureLab/profile).",
          link="https://man7.org/linux/man-pages/man1/mkdir.1.html",
          linkText="mkdir man page"),
    check("board model exposed", fileExists("/sys/class/dmi/id/product_name"),
          hint="This box exposes its model via DMI (/sys/class/dmi/id/product_name); "
               "a Jetson uses /proc/device-tree/model. Re-run the Part 2 model cell.",
          link="https://developer.nvidia.com/embedded/jetson-linux",
          linkText="NVIDIA Jetson Linux"),
    check("CPU readable", commandSucceeds(["nproc"]),
          hint="nproc should exist on any Linux system - re-run the Part 3 cells and "
               "read the error.",
          link="https://man7.org/linux/man-pages/man1/nproc.1.html",
          linkText="nproc man page"),
    check("memory readable", commandSucceeds(["free", "-h"]),
          hint="free should exist on any Linux system - re-run the Part 4 cell.",
          link="https://man7.org/linux/man-pages/man1/free.1.html",
          linkText="free man page"),
    check("disk readable", commandSucceeds("df -h /"),
          hint="df should exist on any Linux system - re-run the Part 6 cell.",
          link="https://man7.org/linux/man-pages/man1/df.1.html",
          linkText="df man page"),
], successNote="You know the node's identity, CPU, memory, GPU, and storage - the raw "
               "constraints your design must live within.",
   docLink="https://developer.nvidia.com/embedded/jetson-linux",
   docLinkText="NVIDIA Jetson Linux")

---
## Part 7 · Watch Live System Load

Static specs tell you the limits; live monitoring tells you the current load. On Jetson, the main tool is `tegrastats` — it prints a new line about once per second **forever**, so it belongs in a terminal, not a notebook cell.

> **New term — `tegrastats`:** NVIDIA's built-in live monitor for Jetson devices. It is the Jetson equivalent of watching `top`, but it also reports GPU and power/thermal data.

**[Notebook cell]** Write the watcher script (it falls back to `top` if `tegrastats` is unavailable):

In [ ]:
%%writefile watchTegrastats.sh
#!/usr/bin/env bash
set -uo pipefail

echo "Live system + GPU monitor - press Ctrl-C to quit."
if command -v nvidia-smi >/dev/null; then
  nvidia-smi dmon        # discrete GPU (this GB10 / servers): util, power, temp stream
elif command -v tegrastats >/dev/null; then
  tegrastats             # Jetson (Tegra SoC): the integrated-GPU telemetry tool
else
  echo "no GPU telemetry tool found - falling back to top."
  top
fi

In [ ]:
!chmod +x watchTegrastats.sh

**[Terminal]** Run it, watch a few lines, then stop with **Ctrl-C**:

```bash
~/edgeArchitectureLab/watchTegrastats.sh
```

**You should see:** a new line printed about once per second, each a long single line of `KEY value` pairs — RAM usage, per-core CPU load, GPU usage (`GR3D_FREQ`), and temperatures.

> Everything `tegrastats` shows is **whole-device** load, shared across all students. When you design for this node, you design for whatever capacity is left after other workloads — not for an empty device.

---
## Part 8 · Check Power and Thermal Limits

Edge nodes are power- and heat-constrained, which directly limits sustained performance.

**[Notebook cell]** If `nvpmodel` is available, check the current power mode. The power mode caps how many cores and how much clock speed the device may use — a lower-power mode means less compute available for your services:

> **📟 On a Jetson:** power management differs by class. A Jetson exposes fixed **power modes** via `nvpmodel` (e.g. 15W/30W/MAXN) that cap cores and clocks to fit a battery/thermal budget. A **GB10** has no `nvpmodel`; you read live power draw and the board's power limit from `nvidia-smi`. Edge-fleet takeaway: on battery-class devices, power mode is a design knob; on a wall-powered box it's mostly a monitoring number.

In [ ]:
# Power. Jetson caps cores/clocks via nvpmodel power modes; a GB10 has no nvpmodel -
# its power draw and limits show up in nvidia-smi instead.
!nvpmodel -q 2>/dev/null || nvidia-smi -q -d POWER 2>/dev/null | grep -iE "Power Draw|Power Limit" || echo "power info not available"


**[Notebook cell]** Read the thermal zones:

In [ ]:
%%bash
for zone in /sys/class/thermal/thermal_zone*/; do
  type=$(cat "$zone/type" 2>/dev/null)
  temp=$(cat "$zone/temp" 2>/dev/null)
  echo "$type: $temp (milli-degrees C)"
done

If a device gets too hot it throttles, lowering clock speeds. This means an edge node's **sustained** performance can be lower than its peak — an architecture must plan for the sustained number, not the peak.

---
## Part 9 · Build a One-Command Profile Report

You explored each resource interactively above. Now combine those same commands into a single script you can re-run any time to capture the device's state. This script — and the one report it writes — is the profiling artifact you save: a reusable engineering tool rather than a one-off.

**[Notebook cell]** Create the script (the original lab uses `nano`; here the notebook writes it for you):

In [ ]:
%%writefile profileNode.sh
#!/bin/bash
# Profiles this edge node and prints a single report.

echo "================================================"
echo " Edge Node Profile"
echo " Generated: $(date)"
echo " By account: $(whoami) on $(hostname)"
echo "================================================"

echo ""
echo "## Identity"
cat /sys/class/dmi/id/product_name 2>/dev/null || cat /proc/device-tree/model 2>/dev/null; echo
nvidia-smi -L 2>/dev/null
uname -a

echo ""
echo "## CPU"
echo "Cores: $(nproc)"
echo "Load average: $(uptime | sed 's/.*load average://')"

echo ""
echo "## Memory"
free -h | grep -E "Mem|Swap"

echo ""
echo "## Storage (root)"
df -h / | tail -1

echo ""
echo "## Accelerator"
ls /dev/nvidia* >/dev/null 2>&1 && echo "NVIDIA device files present" || echo "No NVIDIA device files"
ls /usr/local/cuda >/dev/null 2>&1 && echo "CUDA present" || echo "CUDA not found"

echo ""
echo "## Power"
nvpmodel -q 2>/dev/null | grep -i "power mode" || nvidia-smi -q -d POWER 2>/dev/null | grep -i "Power Limit" || echo "power info not available"

echo ""
echo "================================================"
echo " End of profile"
echo "================================================"


In [ ]:
# Preview the profiling script we just wrote.
showFile("~/edgeArchitectureLab/profileNode.sh")

**[Notebook cell]** Make it executable and run it, saving the result:

In [ ]:
!chmod +x profileNode.sh
!./profileNode.sh | tee profile/node-profile.txt

You now have a single, dated profile of the edge node. Re-running this on any Jetson tells you instantly what you are designing for.

### Checkpoint · Part 9

In [ ]:
checkpoint("Part 9 - one-command profile report", [
    check("profileNode.sh exists", fileExists("~/edgeArchitectureLab/profileNode.sh"),
          hint="Re-run the %%writefile profileNode.sh cell in Part 9.",
          link="https://www.gnu.org/software/bash/manual/bash.html",
          linkText="Bash manual"),
    check("script runs cleanly",
          commandSucceeds("bash ~/edgeArchitectureLab/profileNode.sh"),
          hint="The script errors when run - re-run the %%writefile cell to restore "
               "its content.",
          link="https://www.gnu.org/software/bash/manual/bash.html",
          linkText="Bash manual"),
    check("report saved to profile/node-profile.txt",
          fileNonEmpty("~/edgeArchitectureLab/profile/node-profile.txt", minLines=15),
          hint="Re-run the './profileNode.sh | tee profile/node-profile.txt' cell.",
          link="https://man7.org/linux/man-pages/man1/tee.1.html",
          linkText="tee man page"),
    check("report contains the profile sections",
          fileContains("~/edgeArchitectureLab/profile/node-profile.txt", "## Memory"),
          hint="The saved report looks incomplete - re-run the tee cell in Part 9.",
          link="https://man7.org/linux/man-pages/man1/tee.1.html",
          linkText="tee man page"),
], successNote="A dated, re-runnable profile of the node is saved - the raw material "
               "for your design in Part 10.",
   docLink="https://developer.nvidia.com/embedded/jetson-linux",
   docLinkText="NVIDIA Jetson Linux")

---
## Part 10 · Design the Architecture and Prove It Fits

With the numbers in hand, capture the whole design in **one** document: which tier each piece of work belongs to, why it stays on the edge or goes to the cloud, and — the part that turns a diagram into an *engineered* design — a resource budget that proves the device can actually run it. Edge systems are usually layered into three tiers:

```text
Tier 1: Devices & Sensors    -> cameras, temperature/motion sensors (produce raw data)
Tier 2: Edge Node            -> the DGX Spark (processes data near where it is produced)
Tier 3: Cloud / Data Center  -> long-term storage, fleet dashboards, model training
```

**[Notebook cell]** Write the architecture document **template**, then fill in every bracketed part using **your measured numbers** from `profile/node-profile.txt`:

In [ ]:
%%writefile architecture.md
# Edge System Architecture

## Edge Node Under Design
- Device: [model from profile/node-profile.txt]
- CPU cores: [nproc]   Memory: [total from free -h]   Accelerator: [present / not present]

## Tier Map
- Tier 1 (Devices & Sensors): [e.g. USB camera, temperature sensor]
- Tier 2 (Edge Node - the DGX Spark): [what runs on the device]
- Tier 3 (Cloud): [what is sent off-device]

## Placement Decisions (each justified by a real constraint)
- Stays on the edge: [service] because [latency / privacy / bandwidth / offline operation]
- Goes to the cloud: [service] because [storage / heavy training / fleet-wide view]

## Resource Budget (does the design fit?)
Available now: CPU [nproc] cores | Memory total [free -h] | available [free -h] | disk [df -h /]

| Service            | CPU   | Memory  | Notes                |
|--------------------|-------|---------|----------------------|
| Sensor writer      | low   | ~50 MB  | lightweight          |
| Edge processor     | low   | ~80 MB  | Flask app            |
| Time-series DB     | med   | ~300 MB | stores readings      |
| AI inference (GPU) | high  | [?]     | depends on model     |
| Dashboard          | low   | ~80 MB  |                      |

Total planned memory: [sum] vs. available: [from profile] -> fits? [yes / no]
If not, what moves to the cloud or gets removed: [...]

## Data Flow (optional - Mermaid diagram renders on GitHub)
```mermaid
flowchart TD
    cam[Camera / Sensor] --> proc[Edge Processor on DGX Spark]
    proc --> db[(Local Time-Series DB)]
    proc --> ai[AI Inference on GPU]
    ai --> proc
    db --> dash[Local Dashboard]
    proc --> cloud[Cloud: storage and fleet view]
```

In [ ]:
# Preview the template - every [bracketed] item is yours to replace.
showFile("~/edgeArchitectureLab/architecture.md")

**Now edit the document.** In JupyterLab, open the file browser (folder icon), navigate to `edgeArchitectureLab/`, and double-click `architecture.md` — or use **File ▸ Open from Path…** with `edgeArchitectureLab/architecture.md`. Replace every `[bracketed]` placeholder with your measured numbers and your decisions, then save.

Every placement decision is justified by a real constraint — latency, privacy, bandwidth, available compute, or offline operation — and the budget proves the device can run the design. On a shared DGX Spark, remember that other students' workloads also consume the budget. This is the same pipeline you build piece by piece in later labs (sensors, time-series database, AI inference, dashboards, fleet monitoring); designing it now gives you the map.

**[Notebook cell]** Optionally, save your design and profile with Git (set up in the previous lab — the commit fails harmlessly if Git identity is not configured):

> **See it rendered:** to view the document formatted (rather than the raw text the cell above prints), right-click the `architecture.md` tab and choose **Show Markdown Preview**. The Mermaid data-flow diagram renders there on JupyterLab 4.1+, and always renders on GitHub when you push the file.

### Save your design with git

Put your design under version control so you can snapshot it and come back to it. The next cell:

- **`git init`** — start tracking this `edgeArchitectureLab/` folder,
- **`git add`** — stage `architecture.md` and your `profile/` data,
- **`git commit`** — save a snapshot with a message.

It commits with the git name/email you set in **Lab 1** — or a lab default if you haven't set one — so it never stalls on *"Please tell me who you are."* Re-running is safe: if nothing changed it just says so. The last line lists your saved snapshots.

In [ ]:
%cd ~/edgeArchitectureLab
# init = start tracking this folder | add = stage files | commit = save a snapshot
!git init -q
!git add architecture.md profile/
# Commit with your Lab-1 git identity, or a lab default if you have not configured one,
# so this never stalls on "Please tell me who you are".
!git commit -m "Add edge node profile and architecture design" 2>/dev/null || git -c user.name="${USER}" -c user.email="${USER}@cs494.local" commit -m "Add edge node profile and architecture design" 2>/dev/null || echo "Nothing new to commit - your design is already saved."
!echo "--- your saved snapshots ---"; git log --oneline

### Checkpoint · Part 10

The checks below verify the document exists, keeps its required sections, and — the honest part — that you actually replaced the placeholders with your own numbers.

In [ ]:
import pathlib

def placeholdersFilled():
    docPath = pathlib.Path("~/edgeArchitectureLab/architecture.md").expanduser()
    if not docPath.is_file():
        return False, "missing: " + str(docPath)
    docText = docPath.read_text()
    leftover = [marker for marker in
                ("[model from", "[nproc]", "[sum]", "[yes / no]", "[what runs on the device]")
                if marker in docText]
    if leftover:
        return False, "placeholders still unfilled: " + ", ".join(leftover)
    return True, "all key bracketed placeholders replaced"

checkpoint("Part 10 - architecture designed and budgeted", [
    check("architecture.md exists", fileExists("~/edgeArchitectureLab/architecture.md"),
          hint="Re-run the %%writefile architecture.md cell in Part 10.",
          link="https://www.markdownguide.org/basic-syntax/",
          linkText="Markdown basics"),
    check("Tier Map section kept",
          fileContains("~/edgeArchitectureLab/architecture.md", "## Tier Map"),
          hint="Keep the '## Tier Map' heading while editing - re-add it if you "
               "deleted it.",
          link="https://www.markdownguide.org/basic-syntax/#headings",
          linkText="Markdown headings"),
    check("Placement Decisions section kept",
          fileContains("~/edgeArchitectureLab/architecture.md", "## Placement Decisions"),
          hint="Keep the '## Placement Decisions' heading - every service needs a "
               "constraint-based justification.",
          link="https://www.markdownguide.org/basic-syntax/#headings",
          linkText="Markdown headings"),
    check("Resource Budget section kept",
          fileContains("~/edgeArchitectureLab/architecture.md", "## Resource Budget"),
          hint="Keep the '## Resource Budget' heading - the budget is what proves the "
               "design fits the node.",
          link="https://www.markdownguide.org/basic-syntax/#headings",
          linkText="Markdown headings"),
    check("placeholders filled with your measurements", placeholdersFilled,
          hint="Open architecture.md in the JupyterLab editor and replace every "
               "[bracketed] item with your numbers from profile/node-profile.txt, "
               "then save and re-run this checkpoint.",
          link="https://jupyterlab.readthedocs.io/en/stable/user/file_editor.html",
          linkText="JupyterLab file editor"),
], successNote="Design captured: tiers mapped, placements justified, and the budget "
               "shows it fits the measured node.",
   docLink="https://docs.docker.com/get-started/overview/",
   docLinkText="Next: Docker overview")

---
## Part 11 · Set Your Assigned Docker Port

You have a design on paper. The rest of this lab turns some of those Tier-2 services into real running containers with **Docker** — the tool that packages a service with everything it needs and runs it in isolation on the edge node.

> **Why Docker for the edge:** an *image* bundles the app and its dependencies into one portable artifact, and a *container* runs it in isolation. That is exactly what an edge node needs to host several independent services (sensor, processor, database, dashboard, AI inference) side by side without them interfering. These are the services you just budgeted in Part 10.

Docker resources run through the same Docker daemon on the DGX Spark, so host ports are shared across the whole device — each student must use a **unique** port. Your `PORT` was already derived and exported by the setup cell at the top (and written to `labEnv.sh`, so terminals agree too).

**[Notebook cell]** Check it, then create a Docker working folder under your lab folder:

In [ ]:
!echo "PORT=$PORT"
!mkdir -p ~/edgeArchitectureLab/docker
%cd ~/edgeArchitectureLab/docker

---
## Part 12 · Verify Docker Works on the DGX Spark

**[Notebook cell]** Show the Docker and Compose versions, then verify with `hello-world`:

In [ ]:
dockerVersions()

**You should see:** version numbers, then confirmation that `hello-world` ran. That message means your account can build and run containers on the DGX Spark.

> **New term — image vs. container:** an *image* is a packaged, read-only blueprint (like a class); a *container* is a running instance of an image (like an object). `docker run` turns an image into a container.

> **If it doesn't work:** `permission denied while trying to connect to the Docker daemon` means your account isn't in the `docker` group — ask your instructor. `Cannot connect to the Docker daemon` means Docker isn't running on the DGX Spark.

The original lab then opens an interactive shell (`docker run --rm -it ubuntu bash`) and runs `cat /etc/os-release` inside. That needs a TTY, so from a **notebook cell** we get the same result non-interactively — for the real interactive experience, run `docker run --rm -it ubuntu bash` in a **[Terminal]**:

In [ ]:
!docker run --rm ubuntu cat /etc/os-release

This shows that a container is an isolated environment running on the edge device.

### Checkpoint · Part 12

In [ ]:
checkpoint("Part 12 - Docker verified on the DGX Spark", [
    check("docker daemon reachable", dockerDaemonUp(),
          hint="Docker is not reachable from your account - ask your instructor to add "
               "you to your rootless Podman socket (staff provision it).",
          link="https://docs.docker.com/engine/daemon/troubleshoot/",
          linkText="Docker daemon troubleshooting"),
    check("compose available", composeAvailable(),
          hint="docker compose (v2) is missing - Parts 15 and 17 need it.",
          link="https://docs.docker.com/compose/install/",
          linkText="Installing Compose"),
    check("hello-world image pulled", imageExists("hello-world"),
          hint="Re-run the dockerVersions() cell - it runs hello-world for you.",
          link="https://docs.docker.com/get-started/introduction/",
          linkText="Docker getting started"),
    check("ubuntu container ran", imageExists("ubuntu"),
          hint="Re-run the 'docker run --rm ubuntu cat /etc/os-release' cell.",
          link="https://docs.docker.com/reference/cli/docker/container/run/",
          linkText="docker run reference"),
], successNote="Your account can pull images and run containers on the shared daemon.",
   docLink="https://docs.docker.com/get-started/overview/",
   docLinkText="Docker overview")

---
## Part 13 · Run a Designed Service as a Container

Your Tier-2 design includes a local **dashboard**. Run one as a real container using an NGINX web server, your account name, and your assigned port.

**[Notebook cell]** Any earlier dashboard is removed first, so re-running the cell is safe:

In [ ]:
!docker rm -f ${USER}-archDashboard 2>/dev/null || true
!docker run -d \
  -p $PORT:80 \
  --name ${USER}-archDashboard \
  nginx

> **Docker naming rules — worth reading once, carefully.** Docker does *not* apply one naming convention to everything. It has three namespaces, and only some of them accept capital letters.
>
> **Image names must be lowercase.** `docker build -t ${USER}-nvidia-check .` works. `docker build -t ${USER}-nvidiaCheck .` fails with `invalid reference format: repository name must be lowercase`. The same rule applies to the Compose **project** name you pass to `-p`, and to any Compose service with a `build:` section — Compose derives that image's name from `<project>-<service>`, so an uppercase service name would produce an illegal image name.
>
> **Container names may contain capitals.** That is why this course names containers in camelCase — `${USER}-archDashboard`, `${USER}-telemetryCollector` — matching the code style used everywhere else. **Volume and network names** follow the container rule, so they are camelCase too: `${USER}-edgeData`.
>
> The consequence is that one lab can correctly contain *both* `docker build -t ${USER}-telemetry-collector .` and `docker run --name ${USER}-telemetryCollector ...`. That is not a typo. The image and the container are different objects, and Docker holds them to different rules. If a build ever fails with *"repository name must be lowercase,"* an uppercase letter has crept into an image tag or a project name.

Open the dashboard in a browser using the DGX Spark's address and your assigned port:

```text
http://<thor-address>:<your-port>
```

**[Notebook cell]** See it in the container table and read its logs:

In [ ]:
import os
dockerPs(namePattern=os.environ.get("USER", "student01") + "-archDashboard")
dockerLogs(os.environ.get("USER", "student01") + "-archDashboard", tail=10)

### Checkpoint · Part 13

Run this **while the dashboard is still up** — the practice cell below stops it.

In [ ]:
import os
userName = os.environ.get("USER", "student01")
dashPort = os.environ.get("PORT", "5001")
checkpoint("Part 13 - dashboard container serving", [
    check("dashboard container running", containerRunning(userName + "-archDashboard"),
          hint="Re-run the 'docker run -d ... nginx' cell in Part 13. If the name is "
               "in use, the cell's docker rm -f line clears it on re-run.",
          link="https://docs.docker.com/reference/cli/docker/container/run/",
          linkText="docker run reference"),
    check("your port is listening", portListening(dashPort),
          hint="Nothing is answering on your PORT - re-run the docker run cell and "
               "check its output for a port conflict (another student's container?).",
          link="https://docs.docker.com/engine/network/#published-ports",
          linkText="Docker published ports"),
    check("NGINX answers over HTTP",
          httpOk("http://" + deviceAddress() + ":" + dashPort, expectText="nginx"),
          hint="The port is open but NGINX is not answering - check "
               "docker logs " + userName + "-archDashboard for errors.",
          link="https://nginx.org/en/docs/",
          linkText="NGINX docs"),
], successNote="One Tier-2 box from your architecture diagram is now actually hosted "
               "on the edge device.",
   docLink="https://docs.docker.com/reference/cli/docker/container/run/",
   docLinkText="docker run reference")

**[Notebook cell]** Practice the basic Docker commands you will use to operate any edge service (this stops and removes the dashboard — re-run the Part 13 run cell if you want it back):

In [ ]:
!docker ps
!docker logs ${USER}-archDashboard
!docker stop ${USER}-archDashboard
!docker rm ${USER}-archDashboard

---
## Part 14 · Add Local Persistence with a Volume

Your design also has services that must **keep data** (the time-series database, sensor logs). A container's own filesystem disappears when the container is removed, so persistent data lives in a **Docker volume**: storage managed by Docker that outlives any single container.

**[Notebook cell]** Create a volume using your account name, write a file into it from one throwaway container, then start a **completely separate** container and read the same file back:

In [ ]:
!docker volume inspect ${USER}-edgeData >/dev/null 2>&1 || docker volume create ${USER}-edgeData
!docker run --rm -v ${USER}-edgeData:/data busybox \
  sh -c 'echo "reading saved on the edge node" > /data/log.txt'
!docker run --rm -v ${USER}-edgeData:/data busybox cat /data/log.txt

**You should see:** the line you wrote, even though the first container is already gone. The data survived because it lives in the volume, not in the container. This is how edge systems retain data locally when cloud access is unavailable — the pattern the telemetry and database labs build on.

### Checkpoint · Part 14

In [ ]:
import os
userName = os.environ.get("USER", "student01")
checkpoint("Part 14 - persistent volume", [
    check("volume exists", volumeExists(userName + "-edgeData"),
          hint="Re-run the 'docker volume create' line in the Part 14 cell.",
          link="https://docs.docker.com/engine/storage/volumes/",
          linkText="Docker volumes"),
    check("data survives across containers",
          outputContains(["docker", "run", "--rm", "-v",
                          userName + "-edgeData:/data", "busybox",
                          "cat", "/data/log.txt"],
                         "reading saved on the edge node"),
          hint="The volume exists but the file is missing - re-run the whole Part 14 "
               "cell so the writer container runs again.",
          link="https://docs.docker.com/engine/storage/volumes/",
          linkText="Docker volumes"),
], successNote="Data written by one container was read by another - persistence lives "
               "in the volume, not the container.",
   docLink="https://docs.docker.com/engine/storage/volumes/",
   docLinkText="Docker volumes")

---
## Part 15 · Multi-Container Apps with Docker Compose

Real edge systems run several containers together — a sensor, a processor, a dashboard. Docker **Compose** describes them in one `compose.yaml` and starts them as a unit on a shared network. You will build a full sensor → processor → dashboard pipeline in **Lab 4** and a device fleet in **Lab 9**; here you meet the tool and the one convention you must carry forward.

**[Notebook cell]** Make a tiny two-service Compose file:

In [ ]:
!mkdir -p ~/edgeArchitectureLab/docker/composeIntro
%cd ~/edgeArchitectureLab/docker/composeIntro

In [ ]:
%%writefile compose.yaml
services:
  web:
    image: nginx
    ports:
      - "${PORT}:80"
  clock:
    image: busybox
    command: sh -c "while true; do date; sleep 5; done"


In [ ]:
# Preview the compose file we just wrote.
showFile("~/edgeArchitectureLab/docker/composeIntro/compose.yaml")

**[Notebook cell]** Start both services under a project name that includes your account (`-d` detaches, so the cell returns), then list them and read the clock's logs:

In [ ]:
!docker compose -p ${USER}-compose-intro up -d
!docker compose -p ${USER}-compose-intro ps

In [ ]:
!docker compose -p ${USER}-compose-intro logs clock

Open `http://<thor-address>:<your-port>` in a browser to see NGINX served by the `web` service.

> **Always pass `-p ${USER}-...`.** Compose groups containers, networks, and volumes by project name. On a shared DGX Spark, the `${USER}` prefix keeps each student's stack separate — you use this in every Compose-based lab that follows.

### Checkpoint · Part 15

Run this **while the stack is up** — the cell after it brings the stack down.

In [ ]:
import os
userName = os.environ.get("USER", "student01")
dashPort = os.environ.get("PORT", "5001")
checkpoint("Part 15 - Compose stack up", [
    check("web service running", containerRunning(userName + "-compose-intro-web-1"),
          hint="Re-run the 'docker compose ... up -d' cell. If the port is busy, stop "
               "your Part 13 dashboard first (both use your PORT).",
          link="https://docs.docker.com/compose/",
          linkText="Docker Compose docs"),
    check("clock service running", containerRunning(userName + "-compose-intro-clock-1"),
          hint="Re-run the up -d cell, then 'docker compose -p ${USER}-compose-intro ps' "
               "to see both services.",
          link="https://docs.docker.com/compose/",
          linkText="Docker Compose docs"),
    check("web answers on your port",
          httpOk("http://" + deviceAddress() + ":" + dashPort, expectText="nginx"),
          hint="The stack is up but NGINX is not answering on your PORT - check "
               "'docker compose -p ${USER}-compose-intro ps' for the port mapping.",
          link="https://docs.docker.com/compose/how-tos/networking/",
          linkText="Compose networking"),
], successNote="Two services started as one unit under your own project name.",
   docLink="https://docs.docker.com/compose/",
   docLinkText="Docker Compose docs")

**[Notebook cell]** Stop everything when done:

In [ ]:
!docker compose -p ${USER}-compose-intro down

---
## Part 16 · Build and Run an NVIDIA GPU Container

The whole point of a Jetson is its GPU — and your design's **AI inference** service needs access to it. GPU access from a container comes from two related but separate things: the **base image** (the software environment inside the container) and the **NVIDIA runtime** (host access to the GPU hardware). This part shows both.

The NVIDIA base image tag must match the DGX Spark's JetPack / L4T software stack. `NVIDIA_IMAGE` was set by the setup cell at the top — **use the image tag the instructor provides** if it differs; an image that does not match the installed stack may fail to run.

**[Notebook cell]** Check the image tag and create the build folder:

In [ ]:
!echo "NVIDIA_IMAGE=$NVIDIA_IMAGE"
!mkdir -p ~/edgeArchitectureLab/docker/nvidiaCheck
%cd ~/edgeArchitectureLab/docker/nvidiaCheck

**[Notebook cell]** Create `gpuCheck.sh` and the `Dockerfile`:

In [ ]:
%%writefile gpuCheck.sh
#!/bin/bash

echo "Running inside an NVIDIA-based container"
echo "Container OS:"
cat /etc/os-release

echo ""
echo "Checking NVIDIA device files:"
ls /dev/nvhost* /dev/nvidia* 2>/dev/null | head || echo "No NVIDIA device files found"

echo ""
echo "Checking CUDA folder:"
ls /usr/local/cuda 2>/dev/null || echo "CUDA folder not found in this image"

echo ""
echo "GPU check complete"


In [ ]:
%%writefile Dockerfile
ARG NVIDIA_IMAGE
FROM ${NVIDIA_IMAGE}

WORKDIR /app

COPY gpuCheck.sh .

RUN chmod +x gpuCheck.sh

CMD ["/app/gpuCheck.sh"]

In [ ]:
# Preview the Dockerfile we just wrote.
showFile("~/edgeArchitectureLab/docker/nvidiaCheck/Dockerfile")

**[Notebook cell]** Build the image, passing the base image as a build argument:

In [ ]:
!docker build \
  --build-arg NVIDIA_IMAGE=$NVIDIA_IMAGE \
  -t ${USER}-nvidia-check .

**[Notebook cell]** Run it with the GPU **denied** (`NVIDIA_VISIBLE_DEVICES=void`), then **granted** (`=all`), and compare — device files under `/dev/nvidia*` appear only when granted:

In [ ]:
!docker run --rm -e NVIDIA_VISIBLE_DEVICES=void ${USER}-nvidia-check

In [ ]:
!docker run --rm -e NVIDIA_VISIBLE_DEVICES=all ${USER}-nvidia-check

- the **base image** controls the software environment inside the container
- the **NVIDIA runtime** controls access to the DGX Spark's GPU hardware from the host

With the runtime enabled you should see additional NVIDIA device files and/or the CUDA folder appear. This is the container form of the placement decision you made in Part 10: AI inference stays on the edge *because* the device has the accelerator this container just reached.

### Checkpoint · Part 16

In [ ]:
import os
userName = os.environ.get("USER", "student01")
checkpoint("Part 16 - NVIDIA GPU container", [
    check("nvidiaCheck image built", imageExists(userName + "-nvidia-check"),
          hint="Re-run the docker build cell in Part 16. If the pull fails, confirm "
               "NVIDIA_IMAGE matches the tag your instructor provided (re-run the "
               "setup cell after editing).",
          link="https://docs.docker.com/reference/cli/docker/buildx/build/",
          linkText="docker build reference"),
    check("GPU reachable from a container", nvidiaRuntimeAvailable(),
          hint="The GPU could not be reached from a container on this "
               "device - ask your instructor.",
          link="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html",
          linkText="NVIDIA Container Toolkit"),
    check("container reaches the GPU",
          outputContains(["docker", "run", "--rm", "-e", "NVIDIA_VISIBLE_DEVICES=all",
                          userName + "-nvidia-check"], "GPU check complete",
                         timeoutSeconds=120),
          hint="Re-run the GPU cell (NVIDIA_VISIBLE_DEVICES=all) and read its output - a base image "
               "that does not match the JetPack/L4T stack can fail here.",
          link="https://developer.nvidia.com/embedded/jetpack",
          linkText="NVIDIA JetPack"),
], successNote="The container reached the DGX Spark's GPU stack - base image and runtime "
               "both in place.",
   docLink="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/docker-specialized.html",
   docLinkText="NVIDIA runtime in Docker")

---
## Part 17 · Run the NVIDIA Container with Docker Compose

Describe the same GPU container in Compose instead of a long `docker run`.

**[Notebook cell]** Stay in the folder and write the Compose file:

In [ ]:
%cd ~/edgeArchitectureLab/docker/nvidiaCheck

In [ ]:
%%writefile compose.yaml
services:
  nvidia-check:
    build:
      context: .
      args:
        NVIDIA_IMAGE: ${NVIDIA_IMAGE}
    environment:
      - NVIDIA_VISIBLE_DEVICES=all       # grants the GPU (nvidia is the default runtime here)
      - NVIDIA_DRIVER_CAPABILITIES=all

In [ ]:
# Preview the compose file we just wrote.
showFile("~/edgeArchitectureLab/docker/nvidiaCheck/compose.yaml")

**[Notebook cell]** The check container exits on its own, so a foreground `up` returns — fine as a cell. Run it, then clean up:

In [ ]:
!docker compose -p ${USER}-nvidia-check up --build

In [ ]:
!docker compose -p ${USER}-nvidia-check down

This shows how NVIDIA-backed edge services can be described in a Compose file — the form your AI inference service takes in the YOLO / Edge AI lab.

### Checkpoint · Part 17

In [ ]:
checkpoint("Part 17 - GPU service described in Compose", [
    check("compose.yaml written",
          fileExists("~/edgeArchitectureLab/docker/nvidiaCheck/compose.yaml"),
          hint="Re-run the %%writefile compose.yaml cell in Part 17.",
          link="https://docs.docker.com/reference/compose-file/",
          linkText="Compose file reference"),
    check("service grants the GPU",
          fileContains("~/edgeArchitectureLab/docker/nvidiaCheck/compose.yaml",
                       "NVIDIA_VISIBLE_DEVICES"),
          hint="The NVIDIA_VISIBLE_DEVICES=all line grants GPU access (nvidia is the default runtime) - re-run the "
               "%%writefile cell to restore it.",
          link="https://docs.docker.com/reference/compose-file/services/#runtime",
          linkText="Compose runtime option"),
    check("build arg passes NVIDIA_IMAGE",
          fileContains("~/edgeArchitectureLab/docker/nvidiaCheck/compose.yaml",
                       "NVIDIA_IMAGE"),
          hint="The build args block must pass NVIDIA_IMAGE through - re-run the "
               "%%writefile cell.",
          link="https://docs.docker.com/reference/compose-file/build/#args",
          linkText="Compose build args"),
], successNote="Same GPU container, now declared as a service - ready for the Edge AI "
               "lab's Compose stacks.",
   docLink="https://docs.docker.com/compose/",
   docLinkText="Docker Compose docs")

---
## Part 18 · Monitor Container Usage Against Your Budget

In Part 10 you wrote a resource budget on paper. Docker lets you check real containers against it.

**[Notebook cell]** See per-container resource usage — `--no-stream` takes one snapshot and returns, so it works as a cell:

In [ ]:
!docker stats --no-stream

You will see CPU, memory, network, and block I/O per container.

> Because the DGX Spark is shared, `docker stats` shows containers from **every** student, not just yours. Find yours by their `${USER}-` name prefix.

**[Notebook cell]** You can also **enforce** a budget — cap a container's CPU and memory the way a constrained edge node would:

In [ ]:
!docker run --rm --memory=128m --cpus=0.5 busybox \
  sh -c 'echo "constrained container"; nproc'

**[Notebook cell]** For the live, continuously-updating view, write a watcher script:

In [ ]:
%cd ~/edgeArchitectureLab

In [ ]:
%%writefile watchStats.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/edgeArchitectureLab/labEnv.sh

echo "Live per-container resource usage - press Ctrl-C to quit."
docker stats

In [ ]:
!chmod +x watchStats.sh

**[Terminal]** Watch live CPU, memory, network I/O, and block I/O while containers run:

```bash
~/edgeArchitectureLab/watchStats.sh
```

This connects Part 10's paper budget to real, enforceable limits: `docker stats` measures what a service actually uses, and `--memory` / `--cpus` cap it so no single service starves the others on the shared edge node. On Jetson you can watch the same load at the system level with `tegrastats` (Part 7).

---
## Part 19 · Connect to the Rest of the Course

You profiled a real edge node, designed a system that fits it, and containerized the first pieces of that design with Docker. Every later lab builds one more piece of the architecture you just mapped — and each one runs as a Docker container the way you practiced here:

```text
Device & Sensor Telemetry   -> the sensors in Tier 1 (containerized collector + volume)
Grafana & Time-Series DB    -> local storage + dashboard on the edge node (Compose stack)
YOLO / Edge AI              -> the GPU inference service (NVIDIA runtime container)
Benchmarking & Evaluation   -> measuring whether the design meets its budget
Optimization & Security     -> making services fit and keeping them safe
Networking Protocols        -> how the tiers talk (MQTT / HTTP / WebSocket)
Fleet Management            -> scaling from one DGX Spark to many
```

The main idea: an edge architecture is driven by the constraints of the device. You measured those constraints firsthand, mapped the work across edge tiers, checked that your design fits the hardware, and packaged the services as containers that run on the node. That constraint-driven mindset — plus Docker as the way to run what you design — is the foundation for everything that follows.

---
## Key Terms

- **Edge node:** the computer that processes data near where it is produced (here, the DGX Spark), sitting between sensors and the cloud.
- **Edge tiers:** the layers of an edge system — devices/sensors → edge node → cloud — each doing the work it is best suited for.
- **Accelerator:** specialized hardware (here, the DGX Spark's GPU) that runs AI/compute far faster than the CPU alone.
- **`tegrastats`:** NVIDIA's live system monitor for Jetson (RAM, CPU, GPU, power, temperature).
- **`nvpmodel`:** the Jetson power-mode tool; the active mode caps how many cores and how much clock speed the device may use.
- **Throttling:** when a device slows itself down because it is too hot, lowering sustained performance below the peak.
- **Resource budget:** an accounting of available CPU/memory/storage versus what your planned services need, used to prove a design fits the device.
- **Docker image / container:** an image is a packaged, read-only blueprint; a container is a running instance of it.
- **Docker volume:** storage managed by Docker that lives outside a container, so data survives after the container is removed.
- **Docker Compose:** a tool that defines and runs several containers together from one `compose.yaml` file.
- **`${USER}` prefix:** your username used to name containers, images, volumes, and Compose projects so your resources don't collide with other students' on the shared device.
- **GPU access:** on this device the NVIDIA runtime is the default, so a container reaches the GPU by setting `NVIDIA_VISIBLE_DEVICES=all` (rather than a named `--runtime nvidia`).

---
## Cleanup

Remove only your own files, containers, images, and volumes.

**[Notebook cell]** Write a single cleanup script (you can run it here or in a **[Terminal]** — it is safe to run any number of times):

In [ ]:
%%writefile cleanup.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/edgeArchitectureLab/labEnv.sh

# Compose projects
( cd ~/edgeArchitectureLab/docker/composeIntro 2>/dev/null && docker compose -p "${USER}-compose-intro" down -v ) || true
( cd ~/edgeArchitectureLab/docker/nvidiaCheck 2>/dev/null && docker compose -p "${USER}-nvidia-check" down ) || true

# Standalone containers, volume, and images
docker rm -f "${USER}-archDashboard" 2>/dev/null || true
docker volume rm "${USER}-edgeData" 2>/dev/null || true
docker rmi "${USER}-nvidia-check" 2>/dev/null || true

echo "Cleanup complete."


In [ ]:
!chmod +x cleanup.sh
!./cleanup.sh

**[Notebook cell]** Your profile and design files are small text documents. Remove the lab folder only if you do not want to keep your work (uncomment to run). If you committed your design with Git and want to keep it, push it to GitHub instead of deleting it (see the SSH, Linux, Git, GitHub lab):

In [ ]:
# !rm -rf ~/edgeArchitectureLab

Only remove your own files, containers, images, and volumes.

### Lab scorecard

In [ ]:
labSummary("System Architecture, Design & Containers")